# Kalamazoo County Treasury — Parcel Data Exploratory Analysis

## Purpose

This notebook provides a concise, reproducible profile of the annual delinquent-parcel export used by the Treasury data project.

It focuses on:

- dataset completeness and basic data quality,
- owner and taxpayer representation,
- repeated names and mailing addresses,
- organization/LLC prevalence, and
- patterns that motivated the project's entity-resolution and association logic.

### Design note

Cleaning logic is **not reimplemented in this notebook**. The notebook calls the project's production extraction and cleaning modules so that exploratory results are based on the same transformations used by the ETL pipeline.

> **Data handling:** Keep source workbooks and executed outputs containing names, addresses, or parcel-level details out of a public repository. This notebook is intended to be committed with outputs cleared.

## 1. Configuration

Update `DATA_FILE`, `SHEET_NAME`, and `TAX_YEAR` when profiling a different annual export.

This notebook is designed to live in the repository's `notebooks/` folder. The setup cell automatically locates the project root, adds it to Python's import path, and resolves the source workbook relative to that root. It will also continue to work if Jupyter is started from the repository root or another subfolder within the repository.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start=None):
    """Locate the repository root from the current notebook location."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "etl_pipeline.py").is_file()
            and (candidate / "data_cleaning.py").is_file()
            and (candidate / "entity_resolution.py").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter from "
        "the repository or one of its subfolders."
    )


PROJECT_ROOT = find_project_root()

# Make the production project modules importable even when this
# notebook is opened from the notebooks/ subfolder.
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

import re

from extraction import extract_data
from data_cleaning import DataCleaner

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

DATA_FILE = PROJECT_ROOT / "2025 Roll - Parcels Export.xlsx"
SHEET_NAME = "Only Delinquent Parcels"
TAX_YEAR = 2025

if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Source workbook not found: {DATA_FILE}. "
        "Place the annual export in the project root or update DATA_FILE."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Data file: {DATA_FILE}")

Project root: C:\Users\mattp\Documents\kzoo_county_project
Data file: C:\Users\mattp\Documents\kzoo_county_project\2025 Roll - Parcels Export.xlsx


## 2. Load and clean the data

The same extraction and cleaning components used by the annual ETL pipeline are used here.

In [2]:
extracted_df = extract_data(
    str(DATA_FILE),
    SHEET_NAME,
)

clean_df = DataCleaner().clean_data(
    extracted_df
)

print(f"Tax year: {TAX_YEAR}")
print(f"Rows: {len(clean_df):,}")
print(f"Columns: {len(clean_df.columns):,}")

Tax year: 2025
Rows: 3,874
Columns: 15


## 3. Dataset structure and completeness

The following tables summarize data types and missingness without exposing individual parcel records.

In [3]:
def missing_count(series):
    if (
        pd.api.types.is_object_dtype(series.dtype)
        or pd.api.types.is_string_dtype(series.dtype)
    ):
        return (
            series
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
    return series.isna().sum()


schema_summary = pd.DataFrame({
    "dtype": clean_df.dtypes.astype(str),
    "missing": [
        missing_count(clean_df[column])
        for column in clean_df.columns
    ],
    "unique_values": [
        clean_df[column]
        .replace("", pd.NA)
        .nunique(dropna=True)
        if (
            pd.api.types.is_object_dtype(clean_df[column].dtype)
            or pd.api.types.is_string_dtype(clean_df[column].dtype)
        )
        else clean_df[column].nunique(dropna=True)
        for column in clean_df.columns
    ],
}, index=clean_df.columns)

schema_summary["non_missing"] = (
    len(clean_df) - schema_summary["missing"]
)

schema_summary["missing_pct"] = (
    schema_summary["missing"] / len(clean_df) * 100
).round(2)

schema_summary[
    ["dtype", "non_missing", "missing", "missing_pct", "unique_values"]
].sort_values(
    ["missing_pct", "unique_values"],
    ascending=[False, False],
)

,dtype,non_missing,missing,missing_pct,unique_values
taxpayer_care_of,str,154,3720,96.02,137
owner_care_of,str,202,3672,94.79,184
owner_name_2,str,217,3657,94.40,196
taxpayer_address,str,1978,1896,48.94,1558
taxpayer_name,str,1979,1895,48.92,1658
tax_send_to_name,str,3844,30,0.77,3271
parcel_number,string,3874,0,0.00,3874
property_address,str,3874,0,0.00,3778
owner_name,str,3874,0,0.00,3299
tax_value,int64,3874,0,0.00,3239


### Core dataset checks

These checks are descriptive rather than replacements for the automated ETL tests.

In [4]:
parcel_blank = (
    clean_df["parcel_number"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

property_address_blank = (
    clean_df["property_address"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

core_checks = pd.Series({
    "row_count": len(clean_df),
    "unique_parcels": (
        clean_df.loc[~parcel_blank, "parcel_number"].nunique()
    ),
    "duplicate_parcel_numbers": (
        clean_df.loc[~parcel_blank, "parcel_number"]
        .duplicated()
        .sum()
    ),
    "missing_parcel_numbers": parcel_blank.sum(),
    "missing_property_addresses": property_address_blank.sum(),
})

core_checks.to_frame("value")

,value
row_count,3874
unique_parcels,3874
duplicate_parcel_numbers,0
missing_parcel_numbers,0
missing_property_addresses,0


## 4. Owner and taxpayer overview

Names in this section are summarized only as aggregate counts. Repeated appearances of a name do **not** by themselves prove that every occurrence represents the same legal person or organization.

In [5]:
name_columns = {
    "owner": "owner_name",
    "owner_name_2": "owner_name_2",
    "owner_care_of": "owner_care_of",
    "taxpayer": "taxpayer_name",
    "taxpayer_care_of": "taxpayer_care_of",
    "tax_send_to": "tax_send_to_name",
}

name_summary_rows = []

for label, column in name_columns.items():
    values = (
        clean_df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    populated = values.ne("")
    name_summary_rows.append({
        "source_role": label,
        "populated_rows": populated.sum(),
        "unique_values": values[populated].nunique(),
    })

name_summary = (
    pd.DataFrame(name_summary_rows)
    .set_index("source_role")
)

name_summary

,populated_rows,unique_values
source_role,,
owner,3874,3299
owner_name_2,217,196
owner_care_of,202,184
taxpayer,1979,1658
taxpayer_care_of,154,137
tax_send_to,3844,3271


### Parcels associated with repeated owner names

This is an exploratory frequency profile, not entity resolution. The production resolver applies additional rules before deciding whether source occurrences belong to the same resolved entity.

In [6]:
owner_name_values = (
    clean_df["owner_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

owner_parcel_counts = (
    clean_df.loc[owner_name_values.ne("")]
    .groupby("owner_name")["parcel_number"]
    .nunique()
)

owner_distribution = (
    owner_parcel_counts
    .value_counts()
    .sort_index()
    .rename_axis("parcels_per_owner_name")
    .reset_index(name="owner_name_count")
)

owner_distribution.head(20)

,parcels_per_owner_name,owner_name_count
0,1,2967
1,2,228
2,3,47
3,4,30
4,5,10
5,6,8
6,7,3
7,8,1
8,9,1
9,10,2


In [7]:
owner_repeat_summary = pd.Series({
    "unique_owner_names": owner_parcel_counts.size,
    "owner_names_on_1_parcel": (owner_parcel_counts == 1).sum(),
    "owner_names_on_2_or_more_parcels": (owner_parcel_counts >= 2).sum(),
    "owner_names_on_5_or_more_parcels": (owner_parcel_counts >= 5).sum(),
    "maximum_parcels_for_one_owner_name": owner_parcel_counts.max(),
})

owner_repeat_summary.to_frame("value")

,value
unique_owner_names,3299
owner_names_on_1_parcel,2967
owner_names_on_2_or_more_parcels,332
owner_names_on_5_or_more_parcels,27
maximum_parcels_for_one_owner_name,21


### Taxpayer-name frequency

In [8]:
taxpayer_name_values = (
    clean_df["taxpayer_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

taxpayer_parcel_counts = (
    clean_df.loc[taxpayer_name_values.ne("")]
    .groupby("taxpayer_name")["parcel_number"]
    .nunique()
)

taxpayer_repeat_summary = pd.Series({
    "unique_taxpayer_names": taxpayer_parcel_counts.size,
    "taxpayer_names_on_1_parcel": (taxpayer_parcel_counts == 1).sum(),
    "taxpayer_names_on_2_or_more_parcels": (taxpayer_parcel_counts >= 2).sum(),
    "maximum_parcels_for_one_taxpayer_name": taxpayer_parcel_counts.max(),
})

taxpayer_repeat_summary.to_frame("value")

,value
unique_taxpayer_names,1658
taxpayer_names_on_1_parcel,1490
taxpayer_names_on_2_or_more_parcels,168
maximum_parcels_for_one_taxpayer_name,20


## 5. Owner / taxpayer differences

A different owner and taxpayer name can be meaningful for research, but it does not by itself imply an ownership relationship between those names.

In [9]:
owner_values = (
    clean_df["owner_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

taxpayer_values = (
    clean_df["taxpayer_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

has_both_names = (
    owner_values.ne("")
    & taxpayer_values.ne("")
)

owner_taxpayer_comparison = clean_df.loc[
    has_both_names,
    ["owner_name", "taxpayer_name"],
].copy()

different_owner_taxpayer = (
    owner_taxpayer_comparison["owner_name"]
    != owner_taxpayer_comparison["taxpayer_name"]
)

pd.Series({
    "rows_with_both_names": len(owner_taxpayer_comparison),
    "same_owner_and_taxpayer_name": (~different_owner_taxpayer).sum(),
    "different_owner_and_taxpayer_name": different_owner_taxpayer.sum(),
    "different_pct": round(different_owner_taxpayer.mean() * 100, 2),
}).to_frame("value")

,value
rows_with_both_names,1979.00
same_owner_and_taxpayer_name,1619.00
different_owner_and_taxpayer_name,360.00
different_pct,18.19


## 6. Shared taxpayer addresses

Shared mailing addresses were one of the patterns investigated during development of the association rules. This section reports aggregate address reuse rather than displaying address values.

In [10]:
taxpayer_address_values = (
    clean_df["taxpayer_address"]
    .fillna("")
    .astype(str)
    .str.strip()
)

taxpayer_address_profile = (
    clean_df.loc[taxpayer_address_values.ne("")]
    .groupby("taxpayer_address")
    .agg(
        unique_owner_names=("owner_name", "nunique"),
        unique_taxpayer_names=("taxpayer_name", "nunique"),
        unique_parcels=("parcel_number", "nunique"),
    )
)

shared_taxpayer_address_summary = pd.Series({
    "unique_taxpayer_addresses": len(taxpayer_address_profile),
    "addresses_with_multiple_owner_names": (
        taxpayer_address_profile["unique_owner_names"] > 1
    ).sum(),
    "addresses_with_multiple_taxpayer_names": (
        taxpayer_address_profile["unique_taxpayer_names"] > 1
    ).sum(),
    "addresses_used_by_multiple_parcels": (
        taxpayer_address_profile["unique_parcels"] > 1
    ).sum(),
})

shared_taxpayer_address_summary.to_frame("value")

,value
unique_taxpayer_addresses,1558
addresses_with_multiple_owner_names,126
addresses_with_multiple_taxpayer_names,115
addresses_used_by_multiple_parcels,218


In [11]:
address_reuse_distribution = (
    taxpayer_address_profile["unique_parcels"]
    .value_counts()
    .sort_index()
    .rename_axis("parcels_per_taxpayer_address")
    .reset_index(name="address_count")
)

address_reuse_distribution.head(20)

,parcels_per_taxpayer_address,address_count
0,1,1340
1,2,139
2,3,38
3,4,19
4,5,8
5,6,5
6,7,3
7,8,2
8,9,1
9,17,2


## 7. Organization and LLC indicators

This is a descriptive keyword-based profile. The production entity-resolution logic contains the authoritative classification rules.

In [12]:
entity_columns = [
    "owner_name",
    "owner_name_2",
    "owner_care_of",
    "taxpayer_name",
    "taxpayer_care_of",
    "tax_send_to_name",
]

organization_patterns = [
    r"\bLLC\b",
    r"\bINC\b",
    r"\bCORP(?:ORATION)?\b",
    r"\bCO(?:MPANY)?\b",
    r"\bLP\b",
    r"\bLTD\b",
    r"\bTRUST\b",
    r"\bBANK\b",
    r"\bCHURCH\b",
    r"\bCOUNTY\b",
    r"\bHOLDINGS\b",
    r"\bUNIVERSITY\b",
    r"\bTOWNSHIP\b",
    r"\bVILLAGE\b",
    r"\bDEPARTMENT\b",
    r"\bFOUNDATION\b",
    r"\bESTATE(?:S)?\b",
]

organization_pattern = "|".join(organization_patterns)

stacked_names = (
    clean_df[entity_columns]
    .stack()
    .dropna()
    .astype(str)
    .str.strip()
)

organization_names = stacked_names[
    stacked_names.str.contains(
        organization_pattern,
        case=False,
        regex=True,
        na=False,
    )
]

llc_names = stacked_names[
    stacked_names.str.contains(
        r"\bLLC\b",
        case=False,
        regex=True,
        na=False,
    )
]

pd.Series({
    "source_name_occurrences_matching_org_keywords": len(organization_names),
    "unique_names_matching_org_keywords": organization_names.nunique(),
    "source_name_occurrences_containing_llc": len(llc_names),
    "unique_names_containing_llc": llc_names.nunique(),
}).to_frame("value")

,value
source_name_occurrences_matching_org_keywords,1843
unique_names_matching_org_keywords,583
source_name_occurrences_containing_llc,1284
unique_names_containing_llc,361


## 8. Summary

The exploratory patterns in this notebook explain why the final project uses a dedicated entity-resolution layer rather than treating raw names as unique people or organizations.

In particular:

- repeated names may appear across multiple parcels,
- owner and taxpayer roles may differ,
- mailing addresses can be reused by multiple names or parcels,
- organization naming varies across source records, and
- raw frequency alone is not sufficient evidence for merging identities.

The production system therefore preserves source occurrences, resolves entities conservatively, and stores explicit evidence for cross-entity associations.

### Where to go next

For final entity-resolution outputs and association-level diagnostics, use the companion notebook:

`Entity_Resolution_and_Association_Overview.ipynb`

For authoritative regression validation, use the automated test suite rather than relying on notebook outputs.